In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np

from scenarios import (
    ExperimentScenario,
    SamplingPlan,
    NullSamplingConfig,
    NullFamily,
    generate_null_probabilities,
    sample_multinomial_histograms_for_null,
)
from registry import METHOD_JSD, METHOD_JSD_PREFIX, EXACT_PREFIX, BASELINE_SINGLE
from settings import M_ALTERNATIVE_SAMPLES, M_MONTE_CARLO, rng_global, FloatArray
from io_utils import RESULTS_DIR
from utils import int_logspace_unique
from core import run_experiment_core_for_scenario, make_mn_squared_test, get_decisions_for_histograms
from baselines.multiple_testing import multinull_decisions_holm_batch


In [ ]:
def build_default_scenarios() -> list[ExperimentScenario]:
    """
    Build the four synthetic scenarios used to validate the multi-null decision rule.

    The scenarios span increasing sparsity of the null distributions:

    - Scenario 1 (balanced): a small, dense null family.
    - Scenario 2 (unbalanced): a moderate number of nulls with sparser support.
    - Scenario 3 (border/extreme): more nulls over a larger alphabet with sparser support.
    - Scenario 4 (heterogeneous alpha): the same null geometry as Scenario 3, but with
      heterogeneous per-null significance levels. Baselines are excluded for this scenario,
      since Holm-based correction requires a single family-wise level rather than per-null
      targets.

    For each scenario, M = 10 alternative hypotheses are constructed with target minimum
    Jensen-Shannon distances (mJSd) approximately equal to m / 10 for m = 1, ..., 10, drawn
    from a pool of global Dirichlet candidates and local candidates obtained by mixing each
    null with a Dirichlet draw via a Beta(0.5, 0.5) mixing weight.

    Returns
    -------
    list[ExperimentScenario]
        Scenarios fully configured with `method_plans`.
    """
    # n grid: log-spaced from 1 to 10,000, matched to every method (no method-specific grid).
    shared_n_grid: list[int] = int_logspace_unique(n=29, start=1, stop=10_000)
    shared_mjsd_targets: FloatArray = np.arange(1, 11) / 10.0

    def gen_m_dict(m: int) -> dict[str, int]:
        """Replication counts (null/alt) shared across all methods for a given scenario."""
        return {"null": m, "alt": m}

    method_plans: dict[str, SamplingPlan] = {
        f"{METHOD_JSD_PREFIX}*": SamplingPlan(n_grid=shared_n_grid, m_by_true_kind=gen_m_dict(200)),
        "Chi2-Pearson+Holm": SamplingPlan(n_grid=shared_n_grid, m_by_true_kind=gen_m_dict(200)),
        "G-test-LLR+Holm": SamplingPlan(n_grid=shared_n_grid, m_by_true_kind=gen_m_dict(200)),
        "MMD-Gaussian-MC+Holm": SamplingPlan(n_grid=shared_n_grid, m_by_true_kind=gen_m_dict(200)),
    }

    scen_list: list[ExperimentScenario] = []

    # Scenario 1: balanced nulls.
    scen_list.append(
        ExperimentScenario(
            name="Scenario 1 — Balanced",
            null_sampling_config=NullSamplingConfig(
                num_categories=3,
                num_nulls=3,
                family=NullFamily.UNIFORM_DIRICHLET,
                dirichlet_alpha=1.0,
            ),
            alpha_vector=np.full(shape=(3,), fill_value=0.05, dtype=np.float64),
            n_grid=shared_n_grid,
            method_plans=method_plans,
            ignore_baselines=False,
            methods=["Chi2-Pearson+Holm", "G-test-LLR+Holm", "MMD-Gaussian-MC+Holm"],
            cdf_method="mc_multinomial",
            mc_samples=M_MONTE_CARLO,
            mc_seed=int(rng_global.integers(0, 2**31 - 1)),
            mjsd_targets=shared_mjsd_targets,
            alt_dirichlet_alpha=1.0,
            alt_num_candidate_samples=M_ALTERNATIVE_SAMPLES,
        )
    )

    # Scenario 2: unbalanced nulls.
    scen_list.append(
        ExperimentScenario(
            name="Scenario 2 — Unbalanced",
            null_sampling_config=NullSamplingConfig(
                num_categories=10,
                num_nulls=5,
                family=NullFamily.SPARSE_DIRICHLET,
                dirichlet_alpha=0.7,
            ),
            alpha_vector=np.full(shape=(5,), fill_value=0.05, dtype=np.float64),
            n_grid=shared_n_grid,
            method_plans=method_plans,
            ignore_baselines=False,
            methods=["Chi2-Pearson+Holm", "G-test-LLR+Holm", "MMD-Gaussian-MC+Holm"],
            cdf_method="mc_multinomial",
            mc_samples=M_MONTE_CARLO,
            mc_seed=int(rng_global.integers(0, 2**31 - 1)),
            mjsd_targets=shared_mjsd_targets,
            alt_dirichlet_alpha=0.7,
            alt_num_candidate_samples=M_ALTERNATIVE_SAMPLES,
        )
    )

    # Scenario 3: border/extreme, sparser nulls over a larger alphabet.
    scen_list.append(
        ExperimentScenario(
            name="Scenario 3 — Border/Extreme",
            null_sampling_config=NullSamplingConfig(
                num_categories=25,
                num_nulls=8,
                family=NullFamily.SPARSE_DIRICHLET,
                dirichlet_alpha=0.5,
            ),
            alpha_vector=np.full(shape=(8,), fill_value=0.05, dtype=np.float64),
            n_grid=shared_n_grid,
            method_plans=method_plans,
            ignore_baselines=False,
            methods=["Chi2-Pearson+Holm", "G-test-LLR+Holm", "MMD-Gaussian-MC+Holm"],
            cdf_method="mc_multinomial",
            mc_samples=M_MONTE_CARLO,
            mc_seed=int(rng_global.integers(0, 2**31 - 1)),
            mjsd_targets=shared_mjsd_targets,
            alt_dirichlet_alpha=0.5,
            alt_num_candidate_samples=M_ALTERNATIVE_SAMPLES,
        )
    )

    # Scenario 4: same null geometry as Scenario 3, with heterogeneous per-null alpha.
    # Baselines are excluded, since Holm correction requires a single family-wise level.
    scen_list.append(
        ExperimentScenario(
            name="Scenario 4 — Heterogeneous-alpha",
            null_sampling_config=NullSamplingConfig(
                num_categories=25,
                num_nulls=8,
                family=NullFamily.SPARSE_DIRICHLET,
                dirichlet_alpha=0.5,
            ),
            alpha_vector=np.array(
                [0.01, 0.01, 0.02, 0.02, 0.05, 0.05, 0.10, 0.10], dtype=np.float64
            ),
            n_grid=shared_n_grid,
            method_plans=method_plans,
            ignore_baselines=True,
            cdf_method="mc_multinomial",
            mc_samples=M_MONTE_CARLO,
            mc_seed=int(rng_global.integers(0, 2**31 - 1)),
            mjsd_targets=shared_mjsd_targets,
            alt_dirichlet_alpha=0.5,
            alt_num_candidate_samples=M_ALTERNATIVE_SAMPLES,
        )
    )

    return scen_list


In [ ]:
scenarios: list[ExperimentScenario] = build_default_scenarios()

df_scen_list: list[pd.DataFrame] = []
for scen in scenarios:
    df_scen: pd.DataFrame = run_experiment_core_for_scenario(
        scenario=scen,
        include_baselines=True,
        save_histograms=False,
    )
    df_scen_list.append(df_scen)

df_exp: pd.DataFrame = pd.concat(objs=df_scen_list, ignore_index=True)  # noqa
df_exp.to_csv(path_or_buf=RESULTS_DIR / "experiment.csv", index=False)

df_exp.head()

In [ ]:
# Amortised per-decision runtime on Scenario 3 (k=25, L=8).
#
# For a batch of `batch_size` histograms at each of a few representative sample sizes,
# times each method's decision function and reports the average wall-clock time per decision.
# This is the relevant cost metric for any application that processes many observations at
# the same n, since MNSquared and MMD-Gaussian-MC+Holm both pre-compute their null CDF once
# per n and then decide each histogram at O(L * k) cost.
import time

scenario_3 = scenarios[2]
assert scenario_3.name.startswith("Scenario 3")

batch_size = 300
representative_ns = [50, 300, 1000, 3000]

null_p_s3: FloatArray = generate_null_probabilities(config=scenario_3.null_sampling_config, rng=rng_global)
alpha_vec_s3: FloatArray = scenario_3.alpha_vector
alpha_global_s3: float = float(np.max(alpha_vec_s3))

runtime_rows: list[dict] = []

for n in representative_ns:
    histograms = sample_multinomial_histograms_for_null(
        base_probabilities=null_p_s3[0], num_observations=n, num_histograms=batch_size, rng=rng_global
    )

    # MNSquared: amortise the null CDF pre-computation, then time only the decision step.
    test = make_mn_squared_test(
        null_probabilities=null_p_s3,
        alpha_vector=alpha_vec_s3,
        evidence_size=n,
        cdf_method=scenario_3.cdf_method,
        mc_samples=scenario_3.mc_samples,
        seed=int(rng_global.integers(0, 2**31 - 1)),
    )
    t0 = time.perf_counter()
    get_decisions_for_histograms(test=test, histograms=histograms)
    t_mn2 = (time.perf_counter() - t0) / batch_size
    runtime_rows.append({"method": METHOD_JSD, "n": n, "ms_per_decision": t_mn2 * 1e3})

    # Holm-corrected baselines: Chi2, G-test, and the Monte Carlo-calibrated Gaussian kernel.
    for method_name in ("Chi2-Pearson+Holm", "G-test-LLR+Holm", "MMD-Gaussian-MC+Holm"):
        pval_fn = BASELINE_SINGLE[method_name]
        t0 = time.perf_counter()
        multinull_decisions_holm_batch(
            histograms=histograms,
            null_probabilities=null_p_s3,
            alpha_global=alpha_global_s3,
            single_null_pvalue_fn=pval_fn,
            show_progress=False,
        )
        t_method = (time.perf_counter() - t0) / batch_size
        runtime_rows.append({"method": method_name, "n": n, "ms_per_decision": t_method * 1e3})

df_runtime = pd.DataFrame(runtime_rows)
df_runtime.to_csv(RESULTS_DIR / "scenario_3_runtime.csv", index=False)
df_runtime.pivot(index="n", columns="method", values="ms_per_decision")
